**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

**Day 3 — Modules, packages, and project structure**

**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**


**Concept 1: modules — files are objects too**

A <strong>module</strong> <h6>is just a (.py) file. When you write import pandas, Python finds pandas, executes it top to bottom, and hands you back the resulting object. Every name defined in that file becomes an attribute you access with dot notation.</h6>

- <strong>Key Mental Model<strong><h6>
A module is a namespace. Just like a class, it's a container of names. You instantiate classes; you import modules.<h6>

In [3]:
# profiles.py  ← this file IS the module
CLEAN_THRESHOLD = 0.05          # module-level constant

# Anything defined at the top level of the file becomes an attribute of the module object.
# CLEAN_THRESHOLD will be accessible as profiles.CLEAN_THRESHOLD after import.

class ColumnProfile:
    ...
# A class defined at module level becomes profiles.ColumnProfile.    

def summarize(profile):         # module-level function
    for col in profile:
        print(col.summary())

In [ ]:
# main.py  ← importing from your own module
import profiles                         
# Python finds profiles.py, executes it top to bottom (defining the constant, class, and function).
# Returns the whole module object, bound to the name profiles.
# You access everything through profiles.X.

from profiles import ColumnProfile      # pulls one name out of it
from profiles import ColumnProfile, summarize   # pull multiple
from profiles import CLEAN_THRESHOLD as THRESHOLD  # alias it

p = profiles.ColumnProfile(...)         # dot access
p = ColumnProfile(...)                  # direct access after from-import        

<h6>(<strong>import profiles</strong>) gives you the whole module object — you access everything through (<strong>profiles.X</strong>). (<strong>from profiles import X</strong>) binds just that name in your current namespace. Both execute the file exactly once — Python caches it in (<strong>sys.modules</strong>) after the first import, so re-importing is instant and free.

The key mental model: a <strong><u>module is a namespace</u></strong>. It's a container of names, same as a class. The difference is you instantiate classes; you just import modules.

- import profiles
	
You get the whole module object	

use it when you want to keep names grouped and avoid 
collisions

- from profiles import X

You get just X bound in your namespace	

use it when you use X a lot and want to save typing</h6>

<strong> The Core Idea </strong>

<h6>A module is a (.py) file. When you import it, Python runs it once, stores the result, and gives you back an object. Every name defined in the file becomes an attribute of that object. You use modules exactly like you use classes — through dot notation — because they're both namespaces.</h6>

Concept 2: Packages
A package is a folder containing a file named __init__.py. That one file is what turns an ordinary folder into something Python can import as a namespace. Without __init__.py, the folder is just a folder; Python won't treat it as a package.

A package contains subpackages (folders with their own __init__.py) and modules (.py files). The whole tree becomes a nested namespace you navigate with dots.

The __init__.py file serves two purposes:

It runs when the package is imported.

It decides what the package exposes to the outside world.

Point two is the important one. Inside a package, you may have many files, each containing many classes. But callers shouldn't have to know which file holds which class. The __init__.py re-exports a clean, curated list of names — the public API of the package.

Inside __init__.py, imports use a leading dot to mean "look inside this package, not on the global path":

.module — the module next to me.

.subpackage — the subpackage inside me.

Without the leading dot, Python searches the global import path and usually fails. Relative imports only work inside packages; they cannot be used at the top level of a script.

The reason this matters: you can refactor the internals of a package freely, as long as the __init__.py keeps its promises. Callers depend only on the public API. The internal file layout is your private business.

This is exactly how large libraries like LangChain work. When you write from langchain.tools import BaseTool, you are importing from a package's __init__.py, not from some deep internal file. The library author chose what to expose.



<strong>the (__init__.py) file</strong>

In [ ]:

# data_agent/profiler/__init__.py
# This file decides what `from data_agent.profiler import X` exposes
from .column import ColumnProfile, NumericColumn, CategoricalColumn
from .dataset import DatasetProfile


The . at the start means "relative to this package."

So .column = "the column.py file inside the same folder as this __init__.py."

Python opens column.py, runs it, and pulls three names out of it.

Those three names now live as attributes of the profiler package.

Same idea. .dataset = dataset.py in the same folder.

Pulls DatasetProfile into the profiler package namespace.

<strong>Why the Dot Matters</strong>

In [ ]:
from .column import X          # ✅ relative import — looks inside this package
from column import X           # ❌ absolute import — Python searches sys.path, fails

Without the leading ., Python doesn't know you mean "the column.py that lives next to me." It goes looking on the global import path and usually doesn't find it.

Relative imports only work inside packages. You can't use them at the top level of a script.

<strong>The Public API Idea</strong>

After the two lines in __init__.py, callers can write:



In [ ]:
from data_agent.profiler import DatasetProfile   # clean public API

Without ever knowing that DatasetProfile is defined in dataset.py and not, say, in profiler/__init__.py itself.

Internal structure is your business. Public API is the contract.

If tomorrow you move DatasetProfile from dataset.py into data.py, the caller doesn't notice — as long as __init__.py still imports it from wherever it lives now.

<strong>Real-World Analogy</strong>

This is exactly how big libraries work. When you write:



In [ ]:
from langchain.tools import BaseTool

You're not reaching into langchain/tools/base.py or wherever the class actually lives. You're importing from the package's __init__.py, which the library author has decided is the official way to expose BaseTool.

The library author is free to reorganize internal files at any time. Your import statement won't break.

====

Two-Level Package

store/

├── __init__.py

├── products/

│   ├── __init__.py

│   └── item.py

└── users/

    ├── __init__.py
    └── customer.py

In products/item.py, define Item(name, price) with __str__ returning f"{name} (${price})".

In users/customer.py, define Customer(name) with __str__ returning f"Customer: {name}".

In each __init__.py, expose the corresponding class.

In main.py, import both and print them

In [ ]:
# store/products/item.py
class Item:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        return f"{self.name} (${self.price})"

In [ ]:
# store/products/__init__.py
from .item import Item

In [ ]:
# store/users/customer.py
class Customer:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"Customer: {self.name}"
    
# store/users/__init__.py
from .customer import Customer

In [ ]:
# main.py
from store.products import Item
from store.users import Customer

print(Item("Apple", 1.50))     # Apple ($1.50)
print(Customer("Alice"))       # Customer: Alice

Nested subpackages

school/
├── __init__.py
├── people/
│   ├── __init__.py
│   ├── student.py
│   └── teacher.py
└── courses/
    ├── __init__.py
    └── course.py

In [ ]:
#File: school/people/student.py
class Student:
    def __init__(self, name):
        self.name = name

In [ ]:
#File: school/people/teacher.py
class Teacher:
    def __init__(self, name):
        self.name = name

In [ ]:
#File: school/people/__init__.py

from .student import Student
from .teacher import Teacher

In [ ]:
# File: school/courses/course.py

class Course:
    def __init__(self, title):
        self.title = title

In [ ]:
# File: school/courses/__init__.py

from .course import Course

In [ ]:
#File: main.py

from school.people import Student, Teacher
from school.courses import Course

print(Student("Alice").name)   # Alice
print(Teacher("Bob").name)     # Bob
print(Course("Python 101").title)  # Python 101

Concept 3: Project Structure
Project structure is the way files and folders are arranged in a real Python project. There is no single mandatory layout, but the community has converged on a few conventions that scale well.

The most common top-level pieces are:

A source directory — either the project root itself, or a src/ folder inside it. This is where your package lives.

A tests directory — usually tests/, holding test files that mirror the structure of the source.

A pyproject.toml file — the modern, standard configuration file for the project. It declares the project's name, dependencies, Python version, build system, and tool settings (linters, formatters, test runners).

A README file — describes the project.

A LICENSE file — states the legal terms.

The src/ layout is a widely recommended convention: put your package inside src/ rather than at the project root. This prevents a subtle class of bug where the local package shadows the installed one during testing, and it forces you to test your package as it will actually be installed — not as it happens to sit on disk.

Inside the package itself, the general rule is: one concern per module, one concept per file. If a file grows past a few hundred lines, split it. Group related modules into subpackages. Keep __init__.py short — it should expose names, not define them.

Tests typically mirror the source tree. If your package has myapp/users/service.py, you'll usually have tests/users/test_service.py. This makes it obvious where tests for any given module live.

The pyproject.toml file is the single source of truth for how the project is built and run. It replaces older files like setup.py, setup.cfg, and requirements.txt in modern projects.

Nothing about project structure affects how your code runs. It affects how it is maintained. Good structure means a new contributor can find anything in under a minute.

The Core Idea
A module is a file → namespace.

A package is a folder + __init__.py → namespace.

The __init__.py is a gateway: it decides which names the outside world sees.

Callers depend on the public API, not on internal file layout.

You can refactor internals freely as long as __init__.py keeps its promises.

This is why every large Python codebase looks the same at the top: a folder tree, __init__.py at each level, and a small set of names exposed per level. You now understand the pattern behind every well-designed library.